# Fourier Transform & Reciprocal Lattice Explorer

Interactively explore how real-space atom positions and unit-cell geometry
affect reciprocal-space diffraction patterns.

In [10]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import HBox, VBox, Button, FloatSlider, Layout, Label, Output
from IPython.display import display

# ============================================================
#                INITIAL PARAMETERS
# ============================================================
a_init, b_init = 10.0, 10.0
gamma_init = 90.0                    # degrees
nx_real, ny_real = 3, 3              # real-space tiling (3x3)
nx_recip, ny_recip = 50, 50          # repeats for finite crystal (used only for density)
sigma = 0.2                           # Gaussian width in Å (real-space broadening)
atoms_frac_init = np.array([[0.0, 0.0], [0.5, 0.5]])  # two atoms in fractional coords
num_atoms = len(atoms_frac_init)

# Grid size for real-space & FFT
real_grid_size = 512                # 512x512 real-space grid

# Fixed q-window for reciprocal plot
qmax_plot = np.pi


# ============================================================
#                        STATE
# ============================================================
atoms_frac = atoms_frac_init.copy()
a, b, gamma = a_init, b_init, gamma_init
atom_mask = np.ones(num_atoms, dtype=bool)
suspend_callbacks = False


# ============================================================
#                  LATTICE GEOMETRY FUNCTIONS
# ============================================================
def unit_cell_vectors(a_val, b_val, gamma_deg):
    """Return direct lattice vectors a, b in Cartesian coordinates."""
    g = np.deg2rad(gamma_deg)
    a_vec = np.array([a_val, 0.0])
    b_vec = np.array([b_val * np.cos(g), b_val * np.sin(g)])
    return a_vec, b_vec


def frac_to_cart(frac, a_val, b_val, gamma_deg):
    """Convert fractional coordinates (N,2) to Cartesian using current lattice."""
    a_vec, b_vec = unit_cell_vectors(a_val, b_val, gamma_deg)
    return np.outer(frac[:, 0], a_vec) + np.outer(frac[:, 1], b_vec)


def tile_unit_cell(atoms_cart_local, nx_val, ny_val, a_val, b_val, gamma_deg, center=True):
    """Tile atoms over an nx_val × ny_val supercell, optionally centered."""
    a_vec, b_vec = unit_cell_vectors(a_val, b_val, gamma_deg)
    pts = []
    for i in range(nx_val):
        for j in range(ny_val):
            pts.append(atoms_cart_local + i * a_vec + j * b_vec)
    pts = np.vstack(pts) if len(pts) > 0 else np.empty((0,2))

    if center:
        pts -= (1.5 * a_vec + 1.5 * b_vec)

    return pts


# ============================================================
#               REAL-SPACE DENSITY & 2D FFT
# ============================================================
def build_density(points, grid_size=real_grid_size):
    """Build a real-space Gaussian density map over a fixed grid."""
    x = np.linspace(-25, 25, grid_size)
    y = np.linspace(-25, 25, grid_size)
    X, Y = np.meshgrid(x, y)
    rho = np.zeros_like(X)

    if points.size > 0:
        for px, py in points:
            rho += np.exp(-((X - px)**2 + (Y - py)**2) / (2 * sigma**2))

    return rho, x, y


def compute_fft_q(rho, x, y):
    """Compute 2D FFT and return intensity and qx,qy axes."""
    ny, nx = rho.shape
    dx = x[1] - x[0]
    dy = y[1] - y[0]

    F = np.fft.fftshift(np.fft.fft2(rho))
    intensity = np.abs(F)**2

    fx = np.fft.fftshift(np.fft.fftfreq(nx, d=dx))
    fy = np.fft.fftshift(np.fft.fftfreq(ny, d=dy))

    qx = 2 * np.pi * fx
    qy = 2 * np.pi * fy

    return intensity, qx, qy


def crop_q_window(intensity, qx, qy, qmax):
    """Crop the q map to |qx|,|qy| <= qmax."""
    mask_x = np.where(np.abs(qx) <= qmax)[0]
    mask_y = np.where(np.abs(qy) <= qmax)[0]

    if mask_x.size == 0 or mask_y.size == 0:
        return intensity, qx, qy

    ix_min, ix_max = mask_x[0], mask_x[-1]
    iy_min, iy_max = mask_y[0], mask_y[-1]

    return (
        intensity[iy_min:iy_max+1, ix_min:ix_max+1],
        qx[ix_min:ix_max+1],
        qy[iy_min:iy_max+1],
    )


# ============================================================
#                     UNIT CELL OVERLAY
# ============================================================
def draw_unit_cell_overlay(fig, nx_val, ny_val, a_vec, b_vec, opacity=0.5):
    fig.layout.shapes = []
    center_shift = 1.5 * a_vec + 1.5 * b_vec

    for i in range(nx_val):
        for j in range(ny_val):
            o = i * a_vec + j * b_vec - center_shift
            c0 = o
            c1 = o + a_vec
            c2 = o + a_vec + b_vec
            c3 = o + b_vec

            fig.add_shape(
                type="path",
                path=f"M {c0[0]},{c0[1]} L {c1[0]},{c1[1]} L {c2[0]},{c2[1]} L {c3[0]},{c3[1]} Z",
                line=dict(color=f"rgba(255,255,255,{opacity})", width=2),
                fillcolor="rgba(255,255,255,0)"
            )


# ============================================================
#                   FIGURE CREATION (STATIC)
# ============================================================
# Initial real-space density
atoms_cart = frac_to_cart(atoms_frac, a, b, gamma)
active_atoms = atoms_cart[atom_mask]
lattice_points = tile_unit_cell(active_atoms, nx_real, ny_real, a, b, gamma, center=True)
rho, x, y = build_density(lattice_points, grid_size=real_grid_size)

# Initial reciprocal density
recip_intensity_full, qx_full, qy_full = compute_fft_q(rho, x, y)
recip_intensity, qx_axis, qy_axis = crop_q_window(recip_intensity_full, qx_full, qy_full, qmax_plot)

# Create Plotly figures (not widgets)
real_fig = go.Figure()
real_fig.add_trace(go.Heatmap(z=rho, x=x, y=y, colorscale="Viridis", showscale=False))
real_fig.update_layout(width=520, height=520, title="Real Space (3×3 centered)")
real_fig.update_xaxes(range=[-25, 25])
real_fig.update_yaxes(range=[-25, 25], scaleanchor="x")
draw_unit_cell_overlay(real_fig, nx_real, ny_real, *unit_cell_vectors(a, b, gamma))

recip_fig = go.Figure()
recip_fig.add_trace(go.Heatmap(z=recip_intensity, x=qx_axis, y=qy_axis, colorscale="Viridis", showscale=False))
recip_fig.update_layout(width=520, height=520, title="Reciprocal Space")
recip_fig.update_xaxes(range=[-qmax_plot, qmax_plot], title="qₓ (Å⁻¹)")
recip_fig.update_yaxes(range=[-qmax_plot, qmax_plot], title="qᵧ (Å⁻¹)", scaleanchor="x")

# Wrap figures in Output() widgets
real_out = Output()
recip_out = Output()

with real_out:
    display(real_fig)

with recip_out:
    display(recip_fig)


# ============================================================
#                   MAIN UPDATE FUNCTION
# ============================================================
def update_figures():
    global atoms_cart, lattice_points, rho, x, y
    global recip_intensity, qx_axis, qy_axis

    # --- Real space ---
    atoms_cart = frac_to_cart(atoms_frac, a, b, gamma)
    active_atoms = atoms_cart[atom_mask]

    lattice_points = tile_unit_cell(
        active_atoms, nx_real, ny_real, a, b, gamma, center=True
    )

    rho, x, y = build_density(lattice_points, grid_size=real_grid_size)

    # Update real-space figure
    real_fig.data[0].z = rho
    real_fig.data[0].x = x
    real_fig.data[0].y = y

    draw_unit_cell_overlay(real_fig, nx_real, ny_real, *unit_cell_vectors(a, b, gamma))
    real_fig.update_xaxes(range=[-25, 25])
    real_fig.update_yaxes(range=[-25, 25], scaleanchor="x")

    # Force redraw in Output widget
    with real_out:
        real_out.clear_output(wait=True)
        display(real_fig)

    # --- Reciprocal space ---
    recip_intensity_full, qx_full, qy_full = compute_fft_q(rho, x, y)
    recip_intensity, qx_axis, qy_axis = crop_q_window(
        recip_intensity_full, qx_full, qy_full, qmax_plot
    )

    recip_fig.data[0].z = recip_intensity
    recip_fig.data[0].x = qx_axis
    recip_fig.data[0].y = qy_axis

    recip_fig.update_xaxes(range=[-qmax_plot, qmax_plot], title="qₓ (Å⁻¹)")
    recip_fig.update_yaxes(range=[-qmax_plot, qmax_plot], title="qᵧ (Å⁻¹)", scaleanchor="x")

    # Force redraw
    with recip_out:
        recip_out.clear_output(wait=True)
        display(recip_fig)


# ============================================================
#                   UI CONTROLS (SLIDERS)
# ============================================================
sliders = []
slider_boxes = []

for i in range(num_atoms):
    fx = FloatSlider(description=f"Atom {i} f_x", min=0, max=1, step=0.001,
                     value=float(atoms_frac[i, 0]), layout=Layout(width="420px"))
    fy = FloatSlider(description=f"Atom {i} f_y", min=0, max=1, step=0.001,
                     value=float(atoms_frac[i, 1]), layout=Layout(width="420px"))
    sliders.append((fx, fy))

    def make_frac_callback(idx):
        def cb(change):
            global suspend_callbacks
            if suspend_callbacks:
                return
            atoms_frac[idx, 0] = sliders[idx][0].value
            atoms_frac[idx, 1] = sliders[idx][1].value
            update_figures()
        return cb

    fx.observe(make_frac_callback(i), 'value')
    fy.observe(make_frac_callback(i), 'value')

    slider_boxes.append(VBox([fx, fy], layout=Layout(margin="2px 0")))


# Unit cell sliders
a_slider = FloatSlider(description="a (Å)", min=1, max=30, step=0.1, value=a_init)
b_slider = FloatSlider(description="b (Å)", min=1, max=30, step=0.1, value=b_init)
gamma_slider = FloatSlider(description="Gamma (°)", min=30, max=150, step=0.5, value=gamma_init)

def on_cell_change(change):
    global a, b, gamma
    a = a_slider.value
    b = b_slider.value
    gamma = gamma_slider.value
    update_figures()

a_slider.observe(on_cell_change, 'value')
b_slider.observe(on_cell_change, 'value')
gamma_slider.observe(on_cell_change, 'value')


# Buttons
reset_btn = Button(description="Reset", button_style="primary")
def on_reset(btn):
    global a, b, gamma, atoms_frac, atom_mask, suspend_callbacks
    a, b, gamma = a_init, b_init, gamma_init
    atoms_frac = atoms_frac_init.copy()
    atom_mask[:] = True

    suspend_callbacks = True
    a_slider.value = a
    b_slider.value = b
    gamma_slider.value = gamma

    for i in range(num_atoms):
        sliders[i][0].value = atoms_frac[i, 0]
        sliders[i][1].value = atoms_frac[i, 1]

    suspend_callbacks = False
    update_figures()

reset_btn.on_click(on_reset)

toggle_btn = Button(description="Toggle Atom 1")
def on_toggle(btn):
    atom_mask[1] = not atom_mask[1]
    update_figures()

toggle_btn.on_click(on_toggle)


# ============================================================
#                     UI LAYOUT
# ============================================================
controls = VBox([
    Label("Unit cell:"), a_slider, b_slider, gamma_slider,
    HBox([reset_btn, toggle_btn])
])

figures_box = HBox(
    [real_out, recip_out],
    layout=Layout(justify_content="space-between")
)

ui = VBox([
    figures_box,
    HBox([controls, VBox([Label("Atom fractional sliders (0→1):")] + slider_boxes)])
])

display(ui)

